# 02 - Data exploration

This notebook provides a simple first look at each dataset. It displays example documents and a few basic statistics without modifying or preprocessing the data.

Run `01_data_loading.ipynb` before running this notebook.

## 1. Install packages

In [ ]:
%pip install -q "datasets>=3.0,<5" "matplotlib>=3.8,<4" "pymupdf>=1.24,<2"

## 2. Load the downloaded datasets

In [ ]:
from collections import Counter
from io import BytesIO
from pathlib import Path
import json
import random

import fitz
import matplotlib.pyplot as plt
from datasets import load_from_disk
from PIL import Image

SEED = 566
random.seed(SEED)


def is_repo_root(path: Path) -> bool:
    return (path / "README.md").exists() and (path / "notebooks").exists()


def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, current / "poisoned-paperwork"]
    for parent in current.parents:
        candidates.extend([parent, parent / "poisoned-paperwork"])

    for candidate in candidates:
        if is_repo_root(candidate):
            return candidate

    raise RuntimeError("Could not find the local poisoned-paperwork repository.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"

required_paths = [
    DATA_DIR / "sroie",
    DATA_DIR / "cord_v2",
    DATA_DIR / "synthetic_resume",
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Missing dataset folders. Run 01_data_loading.ipynb first:\n"
        + "\n".join(missing)
    )

sroie = load_from_disk(DATA_DIR / "sroie")
cord = load_from_disk(DATA_DIR / "cord_v2")
resume_dir = DATA_DIR / "synthetic_resume"

print("SROIE:", {split: len(data) for split, data in sroie.items()})
print("CORD v2:", {split: len(data) for split, data in cord.items()})
print("Synthetic resumes:", len(list((resume_dir / 'resume_list_pdf').glob('*.pdf'))))

# SROIE

SROIE contains English receipt images with OCR words, bounding boxes, and labeled fields such as company, date, address, and total.

In [ ]:
# Show four deterministic examples from the training split.
sample_indices = random.Random(SEED).sample(range(len(sroie["train"])), 4)
fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for axis, index in zip(axes, sample_indices):
    example = sroie["train"][index]
    total = example.get("entities", {}).get("total", "missing")
    axis.imshow(example["image"])
    axis.set_title(f"Index {index}\nTotal: {total}")
    axis.axis("off")

fig.suptitle("SROIE receipt examples")
plt.tight_layout()
plt.show()

In [ ]:
split_names = list(sroie.keys())
split_counts = [len(sroie[name]) for name in split_names]
word_counts = [
    len(words)
    for split in sroie.values()
    for words in split["words"]
]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(split_names, split_counts, color="steelblue")
axes[0].set_title("Documents per split")
axes[0].set_ylabel("Documents")

axes[1].hist(word_counts, bins=25, color="steelblue", edgecolor="white")
axes[1].set_title("OCR words per receipt")
axes[1].set_xlabel("Number of words")
axes[1].set_ylabel("Receipts")

plt.tight_layout()
plt.show()

# CORD v2

CORD v2 contains photographed Indonesian receipts with structured JSON annotations for menu items, subtotals, and totals.

In [ ]:
def cord_total(example):
    annotation = example["ground_truth"]
    if isinstance(annotation, str):
        annotation = json.loads(annotation)
    total = annotation.get("gt_parse", {}).get("total", {})
    if isinstance(total, list):
        total = total[0] if total else {}
    return total.get("total_price", "missing") if isinstance(total, dict) else "missing"


sample_indices = random.Random(SEED).sample(range(len(cord["train"])), 4)
fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for axis, index in zip(axes, sample_indices):
    example = cord["train"][index]
    axis.imshow(example["image"])
    axis.set_title(f"Index {index}\nTotal: {cord_total(example)}")
    axis.axis("off")

fig.suptitle("CORD v2 receipt examples")
plt.tight_layout()
plt.show()

In [ ]:
cord_split_names = list(cord.keys())
cord_split_counts = [len(cord[name]) for name in cord_split_names]
receipt_heights = []

for split in cord.values():
    for raw_annotation in split["ground_truth"]:
        annotation = json.loads(raw_annotation) if isinstance(raw_annotation, str) else raw_annotation
        height = annotation.get("meta", {}).get("image_size", {}).get("height")
        if height is not None:
            receipt_heights.append(height)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(cord_split_names, cord_split_counts, color="darkorange")
axes[0].set_title("Documents per split")
axes[0].set_ylabel("Documents")

axes[1].hist(receipt_heights, bins=25, color="darkorange", edgecolor="white")
axes[1].set_title("Receipt image heights")
axes[1].set_xlabel("Height in pixels")
axes[1].set_ylabel("Receipts")

plt.tight_layout()
plt.show()

# SyntheticResumeData

SyntheticResumeData contains synthetic resume PDFs and one JSON annotation file per resume. Most resumes are written in Chinese.

In [ ]:
pdf_dir = resume_dir / "resume_list_pdf"
annotation_dir = resume_dir / "result_gt"
pdf_paths = sorted(pdf_dir.glob("*.pdf"), key=lambda path: int(path.stem))
annotation_paths = sorted(annotation_dir.glob("*.json"), key=lambda path: int(path.stem))

sample_paths = random.Random(SEED).sample(pdf_paths, 4)
fig, axes = plt.subplots(1, 4, figsize=(16, 6))

for axis, pdf_path in zip(axes, sample_paths):
    document = fitz.open(pdf_path)
    pixmap = document[0].get_pixmap(matrix=fitz.Matrix(1.2, 1.2), alpha=False)
    image = Image.open(BytesIO(pixmap.tobytes("png")))
    document.close()

    with (annotation_dir / f"{pdf_path.stem}.json").open(encoding="utf-8") as file:
        annotation = json.load(file)
    degrees = [item.get("degreeLevel") for item in annotation.get("education", [])]

    axis.imshow(image)
    axis.set_title(f"Resume {pdf_path.stem}\nDegrees: {degrees}")
    axis.axis("off")

fig.suptitle("Synthetic resume examples - first page")
plt.tight_layout()
plt.show()

In [ ]:
page_counts = []
education_counts = []
degree_counts = Counter()

for annotation_path in annotation_paths:
    with annotation_path.open(encoding="utf-8") as file:
        annotation = json.load(file)
    education = annotation.get("education", [])
    page_counts.append(annotation.get("metadata", {}).get("pages_count", 0))
    education_counts.append(len(education))
    degree_counts.update(
        item.get("degreeLevel")
        for item in education
        if item.get("degreeLevel")
    )

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
page_distribution = Counter(page_counts)
axes[0].bar(sorted(page_distribution), [page_distribution[x] for x in sorted(page_distribution)], color="seagreen")
axes[0].set_title("Pages per resume")
axes[0].set_xlabel("Pages")
axes[0].set_ylabel("Resumes")

axes[1].hist(education_counts, bins=range(0, max(education_counts) + 2), align="left", color="seagreen", edgecolor="white")
axes[1].set_title("Education entries per resume")
axes[1].set_xlabel("Education entries")
axes[1].set_ylabel("Resumes")

plt.tight_layout()
plt.show()

print("Most common degree labels:")
for degree, count in degree_counts.most_common(15):
    print(f"  {degree}: {count}")

## Initial observations to record

After running the notebook, note anything that could affect later modeling: receipt resolution, image quality, missing totals, language differences, multi-page resumes, and inconsistent degree labels. Actual normalization, filtering, split construction, or merging should be implemented later in `03_data_processing.ipynb`.